[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [APIs and JSON](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)

# Your First API Server &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The cell below rebuilds what the notebook set up: the practice API, whose `serve` runs each app, and
FastAPI. Run it first. Each task makes an app of its own, and serving it replaces the one before.


In [1]:
import importlib
import sys
import urllib.request
from pathlib import Path

import requests
from fastapi import FastAPI, HTTPException

PRACTICE_API = "https://raw.githubusercontent.com/johnfisher-ai/Python-Visual-Guides/main/notebooks/apis-and-json/practice_api.py"

if "google.colab" in sys.modules or not Path("practice_api.py").exists():
    urllib.request.urlretrieve(PRACTICE_API, "practice_api.py")    # in Colab, on every run

import practice_api
importlib.reload(practice_api)    # runs the file as it is now, not a copy imported earlier

BASE = practice_api.start()
print("ready:", BASE)


ready: http://127.0.0.1:8765


**1.** A route that only answers.


In [2]:
app = FastAPI()


@app.get("/ping")
def ping():
    return {"status": "ok"}


app_url = practice_api.serve(app)
response = requests.get(f"{app_url}/ping", timeout=10)
print(response.status_code, response.json())


200 {'status': 'ok'}


`200`, and the dictionary as JSON. A route that does nothing but answer, like this one, lets a
program check that a server is up.


**2.** A list, and a 404.


In [3]:
app = FastAPI()


@app.get("/stations/{station_id}/coordinates")
def coordinates(station_id: str):
    if station_id not in practice_api.STATIONS:
        raise HTTPException(status_code=404, detail=f"no station with id {station_id!r}")
    found = practice_api.STATIONS[station_id]
    return [found["latitude"], found["longitude"]]


app_url = practice_api.serve(app)
for station_id in ["tromso", "bodo"]:
    response = requests.get(f"{app_url}/stations/{station_id}/coordinates", timeout=10)
    print(response.status_code, response.json())


200 [69.65, 18.96]
404 {'detail': "no station with id 'bodo'"}


A list is sent as a JSON array, as a dictionary is sent as an object, and the `404` carries the
detail the route raised.


**3.** A float in the query.


In [4]:
app = FastAPI()


@app.get("/convert")
def convert(celsius: float):
    return {"celsius": celsius, "fahrenheit": celsius * 9 / 5 + 32}


app_url = practice_api.serve(app)
for query in ["celsius=-6.3", "celsius=cold"]:
    response = requests.get(f"{app_url}/convert?{query}", timeout=10)
    print(response.status_code, response.json())


200 {'celsius': -6.3, 'fahrenheit': 20.66}
422 {'detail': [{'type': 'float_parsing', 'loc': ['query', 'celsius'], 'msg': 'Input should be a valid number, unable to parse string as a number', 'input': 'cold'}]}


`-6.3` arrived as a float, and `cold` got a `422` whose `msg` says it is not a number.


**4.** A second, optional parameter.


In [5]:
app = FastAPI()


@app.get("/convert")
def convert(celsius: float, digits: int = 1):
    return {"celsius": celsius, "fahrenheit": round(celsius * 9 / 5 + 32, digits)}


app_url = practice_api.serve(app)
for query in ["celsius=-6.3", "celsius=-6.3&digits=3"]:
    print(requests.get(f"{app_url}/convert?{query}", timeout=10).json())


{'celsius': -6.3, 'fahrenheit': 20.7}
{'celsius': -6.3, 'fahrenheit': 20.66}


With no `digits` in the query, the default of 1 applied. `round` keeps at most that many digits, so
3 digits of 20.66 is still 20.66.


**5.** The parameters, as the document describes them.


In [6]:
document = requests.get(f"{app_url}/openapi.json", timeout=10).json()

for parameter in document["paths"]["/convert"]["get"]["parameters"]:
    print(parameter["name"], parameter["schema"]["type"], "| required:", parameter["required"])


celsius number | required: True
digits integer | required: False


OpenAPI calls a `float` a `number` and an `int` an `integer`, the JSON Schema names from the
**Schemas and Validation** notebook. `digits` is not required, because it has a default.


**6.** A title and a description.


In [7]:
app = FastAPI(title="Conversions")


@app.get("/convert")
def convert(celsius: float):
    """Convert a temperature in degrees Celsius to degrees Fahrenheit."""
    return {"celsius": celsius, "fahrenheit": celsius * 9 / 5 + 32}


app_url = practice_api.serve(app)
document = requests.get(f"{app_url}/openapi.json", timeout=10).json()
print(document["info"]["title"], "|", document["paths"]["/convert"]["get"]["description"])


Conversions | Convert a temperature in degrees Celsius to degrees Fahrenheit.


The title is the app's, and the description is the docstring, word for word. Both appear at the top
of `/docs`, over the route they describe.


---

&#8592; **Back to:** [Your First API Server](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/15-your-first-api-server.ipynb)  &nbsp;&middot;&nbsp;  [APIs and JSON Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)
